<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_collection/Player_Profile/Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FotMob API → Premier League Player Profiles → Excel

This Google Colab notebook builds a **Premier League Player Profiles** dataset directly from FotMob web API endpoints and exports the final result to:

**`Player_Profiles.xlsx`**

The final table follows the same structure as the existing `Player_Profiles.csv`:

1. `FotMob_Player_ID`
2. `Opta_ID`
3. `Player_Key`
4. `Player_Canonical`
5. `Profile_Name`
6. `Primary_Position`
7. `Resolved_Profile_Position`
8. `Historical_Broad_Position`
9. `Final_Position`
10. `Final_Position_Source`
11. `Stats_Name_Aliases`
12. `Teams`
13. `Seasons`

## Workflow

The notebook:

1. Gets Premier League completed matches for each selected season.
2. Reads player IDs, Opta IDs, names, teams, positions, and minutes from `matchDetails`.
3. Builds one unique Premier League player universe.
4. Requests each player's FotMob `playerData` profile.
5. Resolves detailed player positions from `positionDescription`.
6. Uses historical match position as a fallback when the profile position cannot be resolved.
7. Exports the completed dataset to Excel.

> FotMob's web API is unofficial and unversioned. Endpoint schemas can change, so the notebook includes validation and failed-request sheets.

## 1. Install and import libraries

The notebook uses:

- `requests` for API calls,
- `pandas` and `numpy` for data processing,
- `tqdm` for progress bars,
- `openpyxl` for Excel formatting.

In [ ]:
!pip -q install openpyxl tqdm

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

## 2. Configuration

FotMob's Premier League league ID is **47**.

The default seasons match the existing project:

- 2023/2024
- 2024/2025
- 2025/2026

### Testing

For a quick test, set:

```python
MAX_MATCHES_PER_SEASON = 5
MAX_PLAYERS = 20
```

For the full dataset, leave both values as `None`.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

LEAGUE_ID = 47
COUNTRY_CODE = "ENG"

SEASONS = [
    "2023/2024",
    "2024/2025",
    "2025/2026",
]

# None = collect all completed matches
MAX_MATCHES_PER_SEASON = None

# None = request profiles for every discovered player
MAX_PLAYERS = None

REQUEST_DELAY_MIN = 0.5
REQUEST_DELAY_MAX = 0.9

MAX_RETRIES = 4
REQUEST_TIMEOUT = 30

OUTPUT_FILE = Path(
    "/content/Player_Profiles.xlsx"
)

print("Seasons:", SEASONS)
print("Output :", OUTPUT_FILE)

## 3. Create a reusable FotMob API session

The notebook first tries the `/api/data/` route and then falls back to `/api/`.

The request helper:

- retries temporary failures,
- handles rate limiting,
- returns parsed JSON,
- raises a clear error after all retries fail.

In [ ]:
# ============================================================
# HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.fotmob.com/",
})

API_BASES = [
    "https://www.fotmob.com/api/data",
    "https://www.fotmob.com/api",
]


def get_fotmob_json(
    endpoint,
    params=None,
    max_retries=MAX_RETRIES,
):
    '''
    Request JSON from a FotMob endpoint with retries.
    '''

    last_error = None

    for base_url in API_BASES:

        url = f"{base_url}/{endpoint}"

        for attempt in range(
            1,
            max_retries + 1,
        ):

            try:

                response = session.get(
                    url,
                    params=params,
                    timeout=REQUEST_TIMEOUT,
                )

                if response.status_code == 200:
                    return response.json()

                if response.status_code in {
                    404,
                    410,
                }:
                    last_error = (
                        f"HTTP {response.status_code} "
                        f"for {response.url}"
                    )
                    break

                if response.status_code == 429:

                    wait_seconds = (
                        10 * attempt
                    )

                    print(
                        "Rate limited (429). "
                        f"Waiting {wait_seconds} seconds..."
                    )

                    time.sleep(
                        wait_seconds
                    )

                    continue

                last_error = (
                    f"HTTP {response.status_code} "
                    f"for {response.url}"
                )

            except requests.RequestException as exc:

                last_error = str(exc)

            time.sleep(
                3 * attempt
            )

    raise RuntimeError(
        "FotMob request failed. "
        f"Last error: {last_error}"
    )

## 4. Test the API connection

This checks both required API resources:

- `leagues` for Premier League fixtures,
- `playerData` for an individual player profile.

The player example is only a connection test and is not used to build the final dataset.

In [ ]:
# ============================================================
# API CONNECTION TEST
# ============================================================

test_league = get_fotmob_json(
    "leagues",
    params={
        "id": LEAGUE_ID,
        "season": SEASONS[0],
        "ccode3": COUNTRY_CODE,
    },
)

test_matches = (
    test_league
    .get("fixtures", {})
    .get("allMatches", [])
)

print(
    "League API connection successful."
)

print(
    "Fixture records returned:",
    len(test_matches),
)


# Erling Haaland FotMob ID
test_profile = get_fotmob_json(
    "playerData",
    params={
        "id": 737066,
        "includeMarketValues": "false",
    },
)

print(
    "Player profile API connection successful."
)

print(
    "Test profile:",
    test_profile.get("name"),
)

## 5. Parse player information from one match

Only the fields needed to construct the Player Profiles dataset are collected from each completed match.

The match data provides:

- FotMob player ID,
- Opta ID,
- player name,
- team,
- season,
- broad playing position,
- minutes played.

The broad position is later used as a fallback when FotMob's profile metadata does not provide a specific position.

In [ ]:
# ============================================================
# HELPERS FOR MATCH DATA
# ============================================================

POSITION_MAP = {
    0: "GK",
    1: "DEF",
    2: "MID",
    3: "FWD",
}


def extract_minutes(player):
    '''
    Find the Minutes played statistic from one playerStats record.
    '''

    for group in player.get(
        "stats",
        [],
    ):

        group_stats = group.get(
            "stats",
            {},
        )

        stat_info = group_stats.get(
            "Minutes played"
        )

        if isinstance(
            stat_info,
            dict,
        ):

            stat = stat_info.get(
                "stat",
                {},
            )

            if isinstance(
                stat,
                dict,
            ):
                return pd.to_numeric(
                    stat.get("value"),
                    errors="coerce",
                )

    return np.nan


def parse_match_player_context(
    match_data,
    season,
):
    '''
    Extract the player fields needed for the profile universe.
    '''

    content = match_data.get(
        "content",
        {},
    )

    lineup = content.get(
        "lineup",
        {},
    )

    player_stats = content.get(
        "playerStats",
        {},
    )

    # Build broad-position lookup from the lineup.
    position_lookup = {}

    for side in [
        "homeTeam",
        "awayTeam",
    ]:

        team_lineup = lineup.get(
            side,
            {},
        )

        for group_name in [
            "starters",
            "subs",
        ]:

            for player in team_lineup.get(
                group_name,
                [],
            ):

                player_id = player.get(
                    "id"
                )

                if player_id is None:
                    continue

                usual_position_id = (
                    player.get(
                        "usualPlayingPositionId"
                    )
                )

                position_lookup[
                    int(player_id)
                ] = POSITION_MAP.get(
                    usual_position_id
                )

    rows = []

    for _, player in player_stats.items():

        if not player.get("stats"):
            continue

        player_id = player.get("id")

        if player_id is None:
            continue

        player_id = int(
            player_id
        )

        broad_position = (
            position_lookup.get(
                player_id
            )
        )

        if broad_position is None:

            broad_position = (
                POSITION_MAP.get(
                    player.get(
                        "usualPosition"
                    )
                )
            )

        rows.append({
            "Season": season,
            "FotMob_Player_ID":
                player_id,
            "Opta_ID":
                player.get("optaId"),
            "Stats_Name":
                player.get("name"),
            "Team":
                player.get("teamName"),
            "Historical_Broad_Position":
                broad_position,
            "Minutes":
                extract_minutes(
                    player
                ),
        })

    return pd.DataFrame(
        rows
    )

## 6. Discover the Premier League player universe from the API

For each selected season, the notebook:

1. requests the league fixture list,
2. keeps completed and non-canceled matches,
3. requests `matchDetails`,
4. extracts the player context required for the Player Profiles table.

A failed-match table is retained for validation.

In [ ]:
# ============================================================
# COLLECT MATCH-LEVEL PLAYER CONTEXT
# ============================================================

all_match_players = []
failed_matches = []
match_collection_summary = []


for season in SEASONS:

    print(
        "\n" + "=" * 72
    )

    print(
        f"SEASON: {season}"
    )

    print(
        "=" * 72
    )

    league_data = get_fotmob_json(
        "leagues",
        params={
            "id": LEAGUE_ID,
            "season": season,
            "ccode3": COUNTRY_CODE,
        },
    )

    all_matches = (
        league_data
        .get("fixtures", {})
        .get("allMatches", [])
    )

    finished_matches = []

    for match in all_matches:

        status = match.get(
            "status",
            {},
        )

        is_finished = (
            status.get("finished")
            is True
        )

        is_cancelled = (
            status.get("cancelled")
            is True
        )

        if (
            is_finished
            and not is_cancelled
        ):
            finished_matches.append(
                match
            )

    # Remove duplicated match IDs.
    unique_matches = {
        match["id"]: match
        for match in finished_matches
    }

    finished_matches = (
        list(
            unique_matches.values()
        )
    )

    finished_matches = sorted(
        finished_matches,
        key=lambda x: x["id"],
    )

    if MAX_MATCHES_PER_SEASON is not None:

        finished_matches = (
            finished_matches[
                :MAX_MATCHES_PER_SEASON
            ]
        )

    season_frames = []
    successful_matches = 0

    for match in tqdm(
        finished_matches,
        desc=f"Matches {season}",
    ):

        match_id = match["id"]

        try:

            match_data = (
                get_fotmob_json(
                    "matchDetails",
                    params={
                        "matchId":
                            match_id
                    },
                )
            )

            match_players = (
                parse_match_player_context(
                    match_data,
                    season,
                )
            )

            if match_players.empty:

                failed_matches.append({
                    "Season":
                        season,
                    "Match_ID":
                        match_id,
                    "Reason":
                        "No player statistics returned",
                })

            else:

                season_frames.append(
                    match_players
                )

                successful_matches += 1

        except Exception as exc:

            failed_matches.append({
                "Season":
                    season,
                "Match_ID":
                    match_id,
                "Reason":
                    str(exc),
            })

        time.sleep(
            random.uniform(
                REQUEST_DELAY_MIN,
                REQUEST_DELAY_MAX,
            )
        )

    if season_frames:

        season_df = pd.concat(
            season_frames,
            ignore_index=True,
            sort=False,
        )

        all_match_players.append(
            season_df
        )

        player_rows = len(
            season_df
        )

    else:

        player_rows = 0

    match_collection_summary.append({
        "Season":
            season,
        "Completed_Matches_Requested":
            len(
                finished_matches
            ),
        "Successful_Matches":
            successful_matches,
        "Player_Appearance_Rows":
            player_rows,
    })


if not all_match_players:

    raise RuntimeError(
        "No player data was collected."
    )


match_players = pd.concat(
    all_match_players,
    ignore_index=True,
    sort=False,
)


failed_matches_df = pd.DataFrame(
    failed_matches
)


match_summary_df = pd.DataFrame(
    match_collection_summary
)


print(
    "\nMATCH COLLECTION SUMMARY"
)

display(
    match_summary_df
)

print(
    "Player appearance rows:",
    f"{len(match_players):,}"
)

print(
    "Unique FotMob players:",
    f"{match_players['FotMob_Player_ID'].nunique():,}"
)

print(
    "Failed matches:",
    f"{len(failed_matches_df):,}"
)

## 7. Build the unique player universe

The match-level data is aggregated to one row per FotMob player.

The notebook creates:

- a stable `Player_Key` such as `FM_737066`,
- one canonical match-stat name,
- all observed match-stat name aliases,
- all Premier League teams observed in the selected seasons,
- all selected seasons in which the player appeared,
- the historical broad position with the most recorded minutes.

In [ ]:
# ============================================================
# HELPER FUNCTIONS FOR PLAYER UNIVERSE
# ============================================================

def first_mode(series):
    '''
    Return the most common non-null value.
    If there is a tie, use the first mode.
    '''

    values = series.dropna()

    if values.empty:
        return np.nan

    modes = values.mode()

    if not modes.empty:
        return modes.iloc[0]

    return values.iloc[0]


def join_unique(series):
    '''
    Join sorted unique non-null strings with " | ".
    '''

    values = sorted(
        {
            str(value).strip()
            for value in series.dropna()
            if str(value).strip()
        }
    )

    return " | ".join(
        values
    )


# ============================================================
# 1. BASIC PLAYER INFORMATION
# ============================================================

player_base = (
    match_players
    .groupby(
        "FotMob_Player_ID",
        as_index=False,
    )
    .agg(
        Opta_ID=(
            "Opta_ID",
            first_mode,
        ),
        Player_Canonical=(
            "Stats_Name",
            first_mode,
        ),
        Stats_Name_Aliases=(
            "Stats_Name",
            join_unique,
        ),
        Teams=(
            "Team",
            join_unique,
        ),
        Seasons=(
            "Season",
            join_unique,
        ),
    )
)


# ============================================================
# 2. HISTORICAL BROAD POSITION BY MINUTES
# ============================================================

position_minutes = (
    match_players
    .dropna(
        subset=[
            "Historical_Broad_Position"
        ]
    )
    .copy()
)


position_minutes[
    "Minutes"
] = pd.to_numeric(
    position_minutes[
        "Minutes"
    ],
    errors="coerce",
).fillna(0)


position_minutes = (
    position_minutes
    .groupby(
        [
            "FotMob_Player_ID",
            "Historical_Broad_Position",
        ],
        as_index=False,
    )[
        "Minutes"
    ]
    .sum()
)


historical_position = (
    position_minutes
    .sort_values(
        [
            "FotMob_Player_ID",
            "Minutes",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .drop_duplicates(
        "FotMob_Player_ID"
    )
    [
        [
            "FotMob_Player_ID",
            "Historical_Broad_Position",
        ]
    ]
)


players = (
    player_base
    .merge(
        historical_position,
        on="FotMob_Player_ID",
        how="left",
    )
)


players[
    "Player_Key"
] = (
    "FM_"
    + players[
        "FotMob_Player_ID"
    ]
    .astype("Int64")
    .astype("string")
)


players = (
    players
    .sort_values(
        "Player_Canonical"
    )
    .reset_index(
        drop=True
    )
)


if MAX_PLAYERS is not None:

    players = (
        players.head(
            MAX_PLAYERS
        )
        .copy()
    )


print(
    "Unique players:",
    f"{len(players):,}"
)

display(
    players.head(20)
)

## 8. Parse detailed FotMob profile positions

FotMob's `playerData` response includes `positionDescription`.

The position resolver follows the same logic used in the existing project:

1. Use a specific primary profile position when available.
2. If the primary label is generic, use the best detailed position.
3. If the profile still cannot be resolved, use the historical broad match position.

The detailed-position fallback prioritizes:

- `isMainPosition = True`,
- then the position with the highest number of occurrences.

In [ ]:
# ============================================================
# PROFILE POSITION PARSER
# ============================================================

GENERIC_PRIMARY_POSITIONS = {
    "defender",
    "midfielder",
    "forward",
    "keeper",
    "coach",
}


def parse_profile_position(
    profile,
):
    '''
    Parse and resolve FotMob positionDescription.
    '''

    position_description = (
        profile.get(
            "positionDescription"
        )
        or {}
    )

    primary = (
        position_description.get(
            "primaryPosition"
        )
        or {}
    )

    positions = (
        position_description.get(
            "positions"
        )
        or []
    )

    primary_position = (
        primary.get(
            "label"
        )
    )

    detailed_positions = []

    for position in positions:

        full_position = (
            position.get(
                "strPos"
            )
            or {}
        )

        label = (
            full_position.get(
                "label"
            )
        )

        if not label:
            continue

        occurrences = (
            position.get(
                "occurences"
            )
        )

        occurrences = (
            pd.to_numeric(
                occurrences,
                errors="coerce",
            )
        )

        if pd.isna(
            occurrences
        ):
            occurrences = 0

        detailed_positions.append({
            "label":
                label,
            "occurrences":
                float(
                    occurrences
                ),
            "is_main_position":
                bool(
                    position.get(
                        "isMainPosition",
                        False,
                    )
                ),
        })

    # Rank detailed positions:
    # main position first, then occurrences.
    if detailed_positions:

        detailed_positions = sorted(
            detailed_positions,
            key=lambda x: (
                x[
                    "is_main_position"
                ],
                x[
                    "occurrences"
                ],
            ),
            reverse=True,
        )

        best_detailed_position = (
            detailed_positions[0][
                "label"
            ]
        )

    else:

        best_detailed_position = None

    # Generic labels are not specific enough.
    primary_is_specific = (
        primary_position is not None
        and str(
            primary_position
        ).strip().lower()
        not in GENERIC_PRIMARY_POSITIONS
    )

    if primary_is_specific:

        resolved_position = (
            primary_position
        )

        source = (
            "Primary Profile Position"
        )

    elif (
        best_detailed_position
        is not None
    ):

        resolved_position = (
            best_detailed_position
        )

        source = (
            "Detailed Position Fallback"
        )

    else:

        resolved_position = None
        source = "Unresolved"

    return {
        "Profile_Name":
            profile.get("name"),
        "Primary_Position":
            primary_position,
        "Resolved_Profile_Position":
            resolved_position,
        "Profile_Position_Source":
            source,
    }

## 9. Request every player profile

This step calls `playerData` once for each unique FotMob player.

The notebook stores failed profile requests separately, allowing you to review incomplete records without losing the successful data.

In [ ]:
# ============================================================
# COLLECT PLAYER PROFILES
# ============================================================

profile_records = []
failed_profiles = []


for row in tqdm(
    players.itertuples(
        index=False
    ),
    total=len(players),
    desc="Player profiles",
):

    player_id = int(
        row.FotMob_Player_ID
    )

    try:

        profile = get_fotmob_json(
            "playerData",
            params={
                "id":
                    player_id,
                "includeMarketValues":
                    "false",
            },
        )

        parsed = (
            parse_profile_position(
                profile
            )
        )

        parsed[
            "FotMob_Player_ID"
        ] = player_id

        profile_records.append(
            parsed
        )

    except Exception as exc:

        failed_profiles.append({
            "FotMob_Player_ID":
                player_id,
            "Player_Canonical":
                row.Player_Canonical,
            "Reason":
                str(exc),
        })

    time.sleep(
        random.uniform(
            REQUEST_DELAY_MIN,
            REQUEST_DELAY_MAX,
        )
    )


profile_metadata = pd.DataFrame(
    profile_records
)


failed_profiles_df = pd.DataFrame(
    failed_profiles
)


print(
    "Profiles collected:",
    f"{len(profile_metadata):,}"
)

print(
    "Failed profiles:",
    f"{len(failed_profiles_df):,}"
)

if not failed_profiles_df.empty:

    display(
        failed_profiles_df.head(20)
    )

## 10. Build the final Player Profiles dataset

The final position uses this priority:

1. `Resolved_Profile_Position`
2. `Historical_Broad_Position`

`Final_Position_Source` records where the final value came from.

The final output is ordered to exactly match the 13-column structure of the existing Player Profiles CSV.

In [ ]:
# ============================================================
# BUILD FINAL PLAYER PROFILES
# ============================================================

player_profiles = (
    players
    .merge(
        profile_metadata,
        on="FotMob_Player_ID",
        how="left",
    )
)


# ------------------------------------------------------------
# Final position
# ------------------------------------------------------------

player_profiles[
    "Final_Position"
] = (
    player_profiles[
        "Resolved_Profile_Position"
    ]
)


profile_resolved_mask = (
    player_profiles[
        "Resolved_Profile_Position"
    ]
    .notna()
)


player_profiles[
    "Final_Position_Source"
] = np.where(
    profile_resolved_mask,
    player_profiles[
        "Profile_Position_Source"
    ],
    np.where(
        player_profiles[
            "Historical_Broad_Position"
        ].notna(),
        "Historical Match Fallback",
        "Unresolved",
    ),
)


fallback_mask = (
    player_profiles[
        "Final_Position"
    ]
    .isna()
)


player_profiles.loc[
    fallback_mask,
    "Final_Position",
] = (
    player_profiles.loc[
        fallback_mask,
        "Historical_Broad_Position",
    ]
)


# ------------------------------------------------------------
# Opta ID type
# ------------------------------------------------------------

player_profiles[
    "Opta_ID"
] = pd.to_numeric(
    player_profiles[
        "Opta_ID"
    ],
    errors="coerce",
).astype("Int64")


# ------------------------------------------------------------
# Exact final structure
# ------------------------------------------------------------

FINAL_COLUMNS = [
    "FotMob_Player_ID",
    "Opta_ID",
    "Player_Key",
    "Player_Canonical",
    "Profile_Name",
    "Primary_Position",
    "Resolved_Profile_Position",
    "Historical_Broad_Position",
    "Final_Position",
    "Final_Position_Source",
    "Stats_Name_Aliases",
    "Teams",
    "Seasons",
]


player_profiles = (
    player_profiles[
        FINAL_COLUMNS
    ]
    .sort_values(
        "Player_Canonical"
    )
    .reset_index(
        drop=True
    )
)


display(
    player_profiles.head(20)
)

## 11. Validate the final dataset

The validation checks:

- total number of players,
- unique FotMob IDs,
- unique player keys,
- missing profile names,
- missing final positions,
- duplicate player IDs,
- final position source distribution.

In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print(
    "=" * 72
)

print(
    "PLAYER PROFILES VALIDATION"
)

print(
    "=" * 72
)


print(
    "\nRows:",
    f"{len(player_profiles):,}"
)

print(
    "Unique FotMob IDs:",
    f"{player_profiles['FotMob_Player_ID'].nunique():,}"
)

print(
    "Unique Player Keys:",
    f"{player_profiles['Player_Key'].nunique():,}"
)

print(
    "Duplicate FotMob IDs:",
    int(
        player_profiles[
            "FotMob_Player_ID"
        ]
        .duplicated()
        .sum()
    )
)

print(
    "Missing Profile_Name:",
    int(
        player_profiles[
            "Profile_Name"
        ]
        .isna()
        .sum()
    )
)

print(
    "Missing Final_Position:",
    int(
        player_profiles[
            "Final_Position"
        ]
        .isna()
        .sum()
    )
)


print(
    "\nFINAL POSITION SOURCE"
)

display(
    player_profiles[
        "Final_Position_Source"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "Players"
    )
    .to_frame()
)


print(
    "\nFINAL POSITION DISTRIBUTION"
)

display(
    player_profiles[
        "Final_Position"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "Players"
    )
    .to_frame()
)

## 12. Export `Player_Profiles.xlsx`

The workbook contains four sheets:

1. **Player Profiles** — final 13-column dataset.
2. **Match Collection Summary** — completed match collection by season.
3. **Failed Matches** — match-level API failures.
4. **Failed Profiles** — player profile API failures.

The final Player Profiles sheet includes filters, a frozen header row, bold headers, and readable column widths.

In [ ]:
# ============================================================
# EXPORT TO EXCEL
# ============================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    player_profiles.to_excel(
        writer,
        sheet_name="Player Profiles",
        index=False,
    )

    match_summary_df.to_excel(
        writer,
        sheet_name=(
            "Match Collection Summary"
        ),
        index=False,
    )

    if failed_matches_df.empty:

        pd.DataFrame(
            columns=[
                "Season",
                "Match_ID",
                "Reason",
            ]
        ).to_excel(
            writer,
            sheet_name="Failed Matches",
            index=False,
        )

    else:

        failed_matches_df.to_excel(
            writer,
            sheet_name="Failed Matches",
            index=False,
        )

    if failed_profiles_df.empty:

        pd.DataFrame(
            columns=[
                "FotMob_Player_ID",
                "Player_Canonical",
                "Reason",
            ]
        ).to_excel(
            writer,
            sheet_name="Failed Profiles",
            index=False,
        )

    else:

        failed_profiles_df.to_excel(
            writer,
            sheet_name="Failed Profiles",
            index=False,
        )


# ============================================================
# EXCEL FORMATTING
# ============================================================

workbook = load_workbook(
    OUTPUT_FILE
)


for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"

    worksheet.auto_filter.ref = (
        worksheet.dimensions
    )

    for header_cell in worksheet[1]:

        header_cell.font = Font(
            bold=True
        )

        header_cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
        )

    for column_index in range(
        1,
        worksheet.max_column + 1,
    ):

        column_letter = (
            get_column_letter(
                column_index
            )
        )

        values = []

        for row_index in range(
            1,
            min(
                worksheet.max_row,
                500,
            ) + 1,
        ):

            value = worksheet.cell(
                row=row_index,
                column=column_index,
            ).value

            if value is not None:

                values.append(
                    len(
                        str(value)
                    )
                )

        if values:

            width = min(
                max(values) + 2,
                45,
            )

        else:

            width = 12

        worksheet.column_dimensions[
            column_letter
        ].width = width


workbook.save(
    OUTPUT_FILE
)


print(
    "=" * 72
)

print(
    "PLAYER PROFILES EXCEL EXPORT COMPLETE"
)

print(
    "=" * 72
)

print(
    "File:",
    OUTPUT_FILE
)

print(
    "Players:",
    f"{len(player_profiles):,}"
)

print(
    "Columns:",
    len(
        player_profiles.columns
    )
)

## 13. Download the Excel file from Google Colab

Run this cell after the export is complete.

In [ ]:
from google.colab import files

files.download(
    str(OUTPUT_FILE)
)